# Entrenar el LoRA de Elena — ai-toolkit (Z-Image) · modo CLI

Entrena un **LoRA de Z-Image** con ai-toolkit por **línea de comandos** (`python run.py`), SIN la
UI de Node (esa se rompía en Colab). Baja el modelo + el training adapter solos y entrena.
Necesita **GPU ≥12 GB** — **L4 o A100** recomendado.

**Cómo va:** poné **L4** (Entorno → Cambiar tipo de entorno) → *Ejecutar todo*. El entrenamiento
corre en la última celda (~1–1.5 hs, imprime progreso y guarda samples). El `.safetensors` sale a
tu Drive: `MyDrive/ai-toolkit/output/elena_lora/`.


## 1) Montar Drive + verificar el dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/ai-toolkit'
DATASET = os.path.join(BASE, 'datasets', 'elena')
OUTPUT  = os.path.join(BASE, 'output')
os.makedirs(DATASET, exist_ok=True); os.makedirs(OUTPUT, exist_ok=True)
imgs = [f for f in os.listdir(DATASET) if f.lower().endswith(('.png','.jpg','.jpeg'))]
txts = [f for f in os.listdir(DATASET) if f.lower().endswith('.txt')]
print('Dataset:', DATASET)
print(f'  imágenes: {len(imgs)} | captions .txt: {len(txts)}')
if len(imgs) == 0:
    print('\n>> FALTA subir el dataset. Copiá a esa carpeta de Drive las 20 elena_*.png + elena_*.txt')
    print('   (están en tu PC en: Produccion visual/assets/elena/lora-dataset-v1/)')
else:
    print('  OK, dataset presente.')


## 2) Clonar ai-toolkit + instalar (torch cu128 + requirements). Tarda unos minutos.


In [ ]:
%cd /content
import os
if not os.path.isdir('/content/ai-toolkit'):
    !git clone https://github.com/ostris/ai-toolkit.git
%cd /content/ai-toolkit
!pip install -q torch==2.9.1 torchvision==0.24.1 torchaudio==2.9.1 --index-url https://download.pytorch.org/whl/cu128
!pip install -q -r requirements.txt
print('\nai-toolkit instalado')


## 3) (opcional) Login a Hugging Face

Solo si la descarga del modelo falla por *gated*. Si anda sin esto, saltealo.
Conseguí un token en huggingface.co/settings/tokens (read).


In [ ]:
# from huggingface_hub import login
# login('hf_XXXXXXXXXXXX')  # descomentá y pegá tu token si hace falta


## 4) Escribir el config del entrenamiento (Z-Image, valores oficiales de ai-toolkit)


In [ ]:
CONFIG = r'''
---
job: extension
config:
  name: "elena_lora"
  process:
    - type: 'sd_trainer'
      training_folder: "/content/drive/MyDrive/ai-toolkit/output"
      device: cuda:0
      trigger_word: "elenacs"
      network:
        type: "lora"
        linear: 16
        linear_alpha: 16
      save:
        dtype: float16
        save_every: 500
        max_step_saves_to_keep: 4
        push_to_hub: false
      datasets:
        - folder_path: "/content/drive/MyDrive/ai-toolkit/datasets/elena"
          caption_ext: "txt"
          caption_dropout_rate: 0.05
          shuffle_tokens: false
          cache_latents_to_disk: true
          resolution: [ 512, 768 ]
      train:
        batch_size: 1
        steps: 2000
        gradient_accumulation_steps: 1
        train_unet: true
        train_text_encoder: false
        gradient_checkpointing: true
        noise_scheduler: "flowmatch"
        timestep_type: "weighted"
        optimizer: "adamw8bit"
        lr: 1e-4
        unload_text_encoder: false
        ema_config:
          use_ema: true
          ema_decay: 0.99
        dtype: bf16
      model:
        name_or_path: "Tongyi-MAI/Z-Image-Turbo"
        arch: "zimage"
        quantize: true
        quantize_te: true
        low_vram: true
        qtype: "qfloat8"
        assistant_lora_path: "ostris/zimage_turbo_training_adapter/zimage_turbo_training_adapter_v2.safetensors"
      sample:
        sampler: "flowmatch"
        sample_every: 500
        width: 768
        height: 768
        prompts:
          - "elenacs, a woman with short curly dark hair, soft smile, at a 1980s office desk in front of a CRT computer, cream blouse, warm dim side lighting"
          - "elenacs, a woman with short curly dark hair, worried expression, at a 1980s office desk"
          - "elenacs, a woman with short curly dark hair, smoking a cigarette, at a 1980s office desk"
        neg: ""
        seed: 42
        walk_seed: true
        guidance_scale: 1
        sample_steps: 8
meta:
  name: "elena_lora"
  version: '1.0'
'''
open('/content/elena_config.yaml','w').write(CONFIG)
print('config escrito en /content/elena_config.yaml')


## 5) Entrenar 🚀

Corre ~1–1.5 hs en L4. Baja el modelo + adapter la 1ª vez (unos minutos), después entrena e imprime
loss + guarda samples cada 500 steps en tu Drive. Al terminar queda el `.safetensors`.


In [ ]:
%cd /content/ai-toolkit
!python run.py /content/elena_config.yaml


## 6) Listo — usar el LoRA

El LoRA final queda en `MyDrive/ai-toolkit/output/elena_lora/` (`.safetensors`).
Enchufalo en **easy-comfy**: subilo a tu carpeta de loras del Colab de ComfyUI (o "Agregar LoRA por
URL"), activá *Usar LoRA*, y en el prompt usá el trigger **`elenacs`**.
Ej: `elenacs, a woman with short curly dark hair, laughing, at a 1980s office desk`.
